# ASL Model Training & Evaluation Dashboard

Visualize training history, confusion matrix, and per-class metrics.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config_loader import load_config, resolve_path

In [ ]:
config = load_config(str(PROJECT_ROOT / 'config' / 'config.yaml'))
eval_report = resolve_path('outputs/evaluation/evaluation_report.json')
history_files = sorted(resolve_path('data/models').rglob('history.json'))

print(f'Evaluation report exists: {eval_report.exists()}')
print(f'Training runs found: {len(history_files)}')

In [ ]:
if history_files:
    with open(history_files[-1]) as f:
        history = json.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss Curves')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    acc = history.get('accuracy', history.get('acc', []))
    val_acc = history.get('val_accuracy', history.get('val_acc', []))
    axes[1].plot(acc, label='Train')
    axes[1].plot(val_acc, label='Val')
    axes[1].set_title('Accuracy Curves')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if eval_report.exists():
    with open(eval_report) as f:
        metrics = json.load(f)

    print('=== Overall Metrics ===')
    for k in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']:
        print(f'{k}: {metrics[k]:.4f}')

    cm = np.array(metrics['confusion_matrix'])
    labels = config['classes']['all']
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()